# Fiddler Quickstart: Google ADK Integration

This notebook demonstrates how to instrument a Google ADK (Agent Development Kit) agent with Fiddler for comprehensive observability.

**What you'll learn:**
- Install and configure `fiddler-adk`
- Instrument an ADK agent with tools
- Run multi-turn conversations
- View traces in the Fiddler UI

**Time to complete:** ~10 minutes

**Prerequisites:**
- Fiddler account with API key and Application ID
- Google Gemini API key or Vertex AI credentials

## Step 1: Install Dependencies

In [ ]:
!pip install -q fiddler-adk google-adk

## Step 2: Configure Credentials

Replace the placeholder values below with your actual credentials.

In [ ]:
import os

# Fiddler credentials
FIDDLER_URL = "https://your-fiddler-instance.com"       # Your Fiddler instance URL
FIDDLER_API_KEY = "your-fiddler-api-key"                # From Fiddler Settings
FIDDLER_APPLICATION_ID = "your-application-uuid"        # From GenAI Applications page

# Google credentials (choose one)
os.environ["GOOGLE_API_KEY"] = "your-gemini-api-key"    # Option A: Gemini API key

# Option B: Vertex AI (uncomment below, comment out GOOGLE_API_KEY above)
# os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
# os.environ["GOOGLE_CLOUD_PROJECT"] = "your-gcp-project"
# os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

## Step 3: Set Up Fiddler Instrumentation

Two lines to enable automatic tracing for all ADK agents.

In [ ]:
from fiddler_otel import FiddlerClient
from fiddler_adk import GoogleADKInstrumentor

client = FiddlerClient(
    api_key=FIDDLER_API_KEY,
    application_id=FIDDLER_APPLICATION_ID,
    url=FIDDLER_URL,
)
GoogleADKInstrumentor(client).instrument()

print("Fiddler instrumentation enabled!")

## Step 4: Define Tools

Create tools that the agent can use. Tool calls are automatically traced.

In [ ]:
def get_weather(city: str) -> dict:
    """Get current weather for a city.

    Args:
        city: The city name to get weather for.

    Returns:
        A dict with temperature and conditions.
    """
    weather_data = {
        "San Francisco": {"temp_f": 62, "condition": "foggy"},
        "New York": {"temp_f": 78, "condition": "sunny"},
        "London": {"temp_f": 58, "condition": "rainy"},
    }
    data = weather_data.get(city, {"temp_f": 72, "condition": "clear"})
    return {"city": city, **data}


def convert_temperature(temp_f: float, to_unit: str) -> dict:
    """Convert temperature between Fahrenheit and Celsius.

    Args:
        temp_f: Temperature in Fahrenheit.
        to_unit: Target unit ('celsius' or 'fahrenheit').

    Returns:
        A dict with the converted temperature.
    """
    if to_unit.lower() == "celsius":
        return {"temp": round((temp_f - 32) * 5 / 9, 1), "unit": "C"}
    return {"temp": temp_f, "unit": "F"}


print("Tools defined: get_weather, convert_temperature")

## Step 5: Create the Agent

Create a Gemini-powered agent with the tools attached.

In [ ]:
from google.adk.agents.llm_agent import Agent

agent = Agent(
    model="gemini-2.5-flash",
    name="weather_agent",
    description="A helpful weather assistant",
    instruction="You help users check the weather. Use the tools available to get weather data and convert temperatures. Be concise.",
    tools=[get_weather, convert_temperature],
)

print(f"Agent '{agent.name}' created with {len(agent.tools)} tools")

## Step 6: Run the Agent

Run a multi-turn conversation. Each turn creates a trace in Fiddler.

In [ ]:
import asyncio
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types


async def run_conversation():
    session_service = InMemorySessionService()
    runner = Runner(
        agent=agent, app_name="weather_demo", session_service=session_service
    )
    session = await session_service.create_session(
        app_name="weather_demo", user_id="notebook_user"
    )

    # Multi-turn conversation
    prompts = [
        "What's the weather in San Francisco?",
        "Convert that temperature to Celsius",
        "How about New York?",
    ]

    for prompt in prompts:
        print(f"\nUser: {prompt}")
        message = types.Content(
            role="user", parts=[types.Part(text=prompt)]
        )

        async for event in runner.run_async(
            user_id="notebook_user",
            session_id=session.id,
            new_message=message,
        ):
            if hasattr(event, "content") and event.content:
                parts = getattr(event.content, "parts", []) or []
                text = "".join(
                    p.text for p in parts if getattr(p, "text", None)
                )
                if text:
                    print(f"Agent: {text}")


await run_conversation()

## Step 7: Flush Traces

Ensure all traces are sent to Fiddler before the notebook ends.

In [ ]:
client.force_flush(timeout_millis=10000)
print("Traces flushed to Fiddler!")

## Step 8: View in Fiddler

Navigate to your Fiddler instance and open your application. In the Trace Explorer you should see:

- **Agent spans** (`invoke_agent`) — one per turn
- **LLM spans** (`call_llm`) — with input prompt, model response, and system instructions
- **Tool spans** (`execute_tool`) — with tool arguments and return values
- **Token counts** — input, output, and reasoning tokens per LLM call

Each turn is grouped by session ID for multi-turn conversation tracking.

## Next Steps

- [Google ADK Integration Guide](https://docs.fiddler.ai/integrations/agentic-ai/google-adk-sdk) — Full documentation
- [Fiddler Evals SDK](https://docs.fiddler.ai/integrations/agentic-ai/evals-sdk) — Evaluate agent quality
- [Google ADK Documentation](https://adk.dev) — Learn more about ADK